# Multi-Strategy Toolpath: AI-Driven Layer-Wise Deposition

**PhD Thesis — Diana Paola Ayala Roldán**

## Core Idea

Different wound layers require different deposition strategies:
- **Deep layers** → Honeycomb (structural integrity, fast fill)
- **Surface layers** → Geodesic (conformal, matches tissue curvature)
- **Transition layers** → Hybrid (honeycomb core + geodesic perimeter)

This notebook demonstrates the **StrategyRouter** that classifies each layer and generates the appropriate toolpath.

**No GPU required** — this is a planning/visualization notebook.

In [ ]:
import sys
import glob
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.patches import RegularPolygon, Patch
from matplotlib.path import Path as MplPath
from pathlib import Path
from mpl_toolkits.mplot3d import Axes3D

# Add repo src to path (works locally and on Kaggle)
REPO_CANDIDATES = [
    str(Path('..') / 'src'),
    *[str(Path(p) / 'src') for p in glob.glob('/kaggle/input/bioprinting-pipeline-multi*')],
    *[str(Path(p) / 'src') for p in glob.glob('/kaggle/input/multistrategy*')],
]
for p in REPO_CANDIDATES:
    if Path(p).joinpath('multistrategy').exists():
        sys.path.insert(0, p)
        print(f'multistrategy found at: {p}')
        break
else:
    print('WARNING: multistrategy not found, imports may fail')

from multistrategy.toolpath.strategy_router import (
    StrategyRouter, StrategyConfig, LayerStrategy, MultiStrategyPlan, LayerPlan
)
from multistrategy.toolpath.honeycomb_planner import plan_honeycomb
from multistrategy.toolpath.geodesic_planner import plan_geodesic
from multistrategy.toolpath.hybrid_planner import plan_hybrid

Path('figures').mkdir(exist_ok=True)
print('Multi-Strategy Toolpath loaded successfully')

## 1. Synthetic Wound Input

We simulate a realistic wound with varying depth — deep center, shallow edges — to demonstrate how the strategy router classifies each layer.

In [ ]:
# Generate synthetic wound boundary and layer amounts
np.random.seed(42)
NUM_RADII = 64
NUM_LAYERS = 6

# Elliptical wound boundary (mm)
angles = np.linspace(0, 2 * np.pi, NUM_RADII, endpoint=False)
a, b = 25.0, 18.0  # semi-axes
radii_mm = (a * b) / np.sqrt((b * np.cos(angles))**2 + (a * np.sin(angles))**2)
radii_mm += np.random.normal(0, 0.8, NUM_RADII)  # slight irregularity
boundary_mm = np.stack([radii_mm * np.cos(angles), radii_mm * np.sin(angles)], axis=1)

# Depth profile (mm) — deeper at center
depth_profile_mm = 8.0 * (1.0 + 0.3 * np.cos(angles * 2)) + np.random.normal(0, 0.3, NUM_RADII)

# Layer amounts: (num_radii, num_layers) — deep layers have high fill, surface layers low
layer_amounts = np.zeros((NUM_RADII, NUM_LAYERS))
for i in range(NUM_LAYERS):
    base_fill = 1.0 - (i / NUM_LAYERS) * 0.85
    spatial_var = 0.8 + 0.2 * np.cos(angles * 2)
    layer_amounts[:, i] = base_fill * spatial_var + np.random.normal(0, 0.03, NUM_RADII)
    layer_amounts[:, i] = np.clip(layer_amounts[:, i], 0.05, 1.0)

print(f'Wound boundary: {NUM_RADII} points, diameter ~{np.ptp(boundary_mm[:, 0]):.1f} x {np.ptp(boundary_mm[:, 1]):.1f} mm')
print(f'Depth range: {depth_profile_mm.min():.1f} – {depth_profile_mm.max():.1f} mm')
print(f'Layer amounts shape: {layer_amounts.shape} (num_radii x num_layers)')
print(f'Mean fill per layer: {[f"{layer_amounts[:, i].mean():.2f}" for i in range(NUM_LAYERS)]}')

## 2. Strategy Router: Layer Classification & Full Plan

In [ ]:
# Configure and run the strategy router
config = StrategyConfig(
    structural_threshold=0.6,
    conformal_threshold=0.3,
)
router = StrategyRouter(config)

# Classify layers
strategies = router.classify_layers(layer_amounts, depth_profile_mm)

# Generate full plan
plan = router.plan(layer_amounts, depth_profile_mm, boundary_mm)

print('\n' + '='*55)
print('  MULTI-STRATEGY LAYER CLASSIFICATION')
print('='*55)
mean_fills = layer_amounts.mean(axis=0)
for i, (strat, lp) in enumerate(zip(strategies, plan.layers)):
    icon = {'HONEYCOMB': '[HEX]', 'GEODESIC': '[GEO]', 'HYBRID': '[HYB]'}[strat.name]
    bar = '#' * int(mean_fills[i] * 25)
    print(f'  Layer {i}: {icon} {strat.name:<10} | fill={mean_fills[i]:.2f} | path={lp.path_length_mm:.0f}mm | {bar}')
print('='*55)
print(f'Strategy breakdown: {plan.strategy_breakdown}')
print(f'Total path length: {plan.total_path_length_mm:.1f} mm')
print(f'Estimated print time: {plan.estimated_print_time_s:.0f} s ({plan.estimated_print_time_s/60:.1f} min)')
print(f'Estimated volume: {plan.estimated_volume_mm3:.2f} mm³')

## 3. Visualization: Strategy Classification Map

In [ ]:
strategy_colors = {
    LayerStrategy.HONEYCOMB: '#FF8C00',
    LayerStrategy.GEODESIC: '#0077BE',
    LayerStrategy.HYBRID: '#8B008B',
}
strategy_labels = {
    LayerStrategy.HONEYCOMB: 'Honeycomb (structural)',
    LayerStrategy.GEODESIC: 'Geodesic (conformal)',
    LayerStrategy.HYBRID: 'Hybrid (transition)',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: Layer amounts heatmap
ax = axes[0]
im = ax.imshow(layer_amounts.T, aspect='auto', cmap='YlOrRd', interpolation='nearest',
               origin='lower')
ax.set_xlabel('Point index (around boundary)')
ax.set_ylabel('Layer (0=deepest)')
ax.set_title('Layer Fill Amounts\n(from Volumetric Decoder)', fontsize=11, fontweight='bold')
fig.colorbar(im, ax=ax, label='Fill fraction')

for i, strat in enumerate(strategies):
    color = strategy_colors[strat]
    ax.text(NUM_RADII + 1, i, f' {strat.name}', va='center', fontsize=7,
            color=color, fontweight='bold')

# Panel 2: Bar chart with strategy coloring
ax = axes[1]
colors = [strategy_colors[s] for s in strategies]
bars = ax.barh(range(NUM_LAYERS), mean_fills, color=colors, edgecolor='k', linewidth=0.5, alpha=0.8)
ax.axvline(config.structural_threshold, color='darkorange', linestyle='--', lw=2,
           label=f'Structural ({config.structural_threshold})')
ax.axvline(config.conformal_threshold, color='steelblue', linestyle='--', lw=2,
           label=f'Conformal ({config.conformal_threshold})')
ax.set_xlabel('Mean fill fraction')
ax.set_ylabel('Layer (0=deepest)')
ax.set_title('Strategy Assignment\n(Threshold-Based Classification)', fontsize=11, fontweight='bold')
ax.set_xlim(0, 1.1)

legend_patches = [Patch(facecolor=c, label=l) for c, l in
                  zip(strategy_colors.values(), strategy_labels.values())]
ax.legend(handles=legend_patches + ax.get_legend_handles_labels()[0],
          fontsize=7, loc='lower right')

plt.tight_layout()
plt.savefig('figures/01_strategy_classification.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/01_strategy_classification.png')

## 4. Per-Layer Toolpath Visualization

In [ ]:
# Visualize actual generated toolpaths from the plan
ncols = 3
nrows = (NUM_LAYERS + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 5 * nrows))
axes = axes.flatten()

bx, by = boundary_mm[:, 0], boundary_mm[:, 1]

for idx, lp in enumerate(plan.layers):
    ax = axes[idx]
    color = strategy_colors[lp.strategy]

    # Wound boundary
    ax.plot(np.append(bx, bx[0]), np.append(by, by[0]), 'k-', linewidth=2, alpha=0.6)
    ax.fill(bx, by, alpha=0.05, color='gray')

    # Plot the planned waypoints (XY projection)
    wp = lp.waypoints_mm
    dep_mask = lp.is_deposition

    # Deposition segments
    dep_wp = wp[dep_mask]
    if len(dep_wp) > 0:
        ax.scatter(dep_wp[:, 0], dep_wp[:, 1], c=np.arange(len(dep_wp)),
                   cmap='viridis', s=8, zorder=3, alpha=0.7)
        ax.plot(dep_wp[:, 0], dep_wp[:, 1], '-', color=color, linewidth=0.8, alpha=0.5)

    # Travel segments (non-deposition)
    travel_wp = wp[~dep_mask]
    if len(travel_wp) > 0:
        ax.scatter(travel_wp[:, 0], travel_wp[:, 1], c='lightgray', s=3, alpha=0.3)

    ax.set_xlim(bx.min() - 3, bx.max() + 3)
    ax.set_ylim(by.min() - 3, by.max() + 3)
    ax.set_aspect('equal')
    ax.set_title(f'Layer {idx}: {lp.strategy.name}\n'
                 f'fill={lp.fill_amount:.2f}, path={lp.path_length_mm:.0f}mm',
                 fontsize=10, fontweight='bold', color=color)
    ax.set_xlabel('X (mm)')
    ax.set_ylabel('Y (mm)')
    ax.grid(True, alpha=0.2)

# Hide unused subplots
for idx in range(NUM_LAYERS, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Per-Layer Toolpath: AI-Selected Strategy\n'
             '(Deep=Honeycomb → Transition=Hybrid → Surface=Geodesic)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/02_per_layer_toolpath.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/02_per_layer_toolpath.png')

## 5. 3D Layer Stack Visualization

In [ ]:
fig = plt.figure(figsize=(14, 8))
ax = fig.add_subplot(111, projection='3d')

for lp in plan.layers:
    color = strategy_colors[lp.strategy]
    wp = lp.waypoints_mm
    dep_mask = lp.is_deposition

    # Draw deposition waypoints in 3D
    dep_wp = wp[dep_mask]
    if len(dep_wp) > 1:
        ax.plot(dep_wp[:, 0], dep_wp[:, 1], dep_wp[:, 2],
                '-', color=color, linewidth=0.8, alpha=0.7)
        ax.scatter(dep_wp[::5, 0], dep_wp[::5, 1], dep_wp[::5, 2],
                   c=color, s=5, alpha=0.5)

    # Label
    ax.text(bx.max() + 5, 0, lp.depth_mm,
            f'L{lp.layer_index}: {lp.strategy.name}',
            fontsize=8, color=color, fontweight='bold')

ax.set_xlabel('X (mm)')
ax.set_ylabel('Y (mm)')
ax.set_zlabel('Height (mm)')
ax.set_title('3D Layer Stack: Multi-Strategy Deposition\n'
             '(Bottom=structural honeycomb, Top=conformal geodesic)',
             fontsize=12, fontweight='bold')
ax.view_init(elev=25, azim=-45)

legend_patches = [Patch(facecolor=strategy_colors[s], label=strategy_labels[s])
                  for s in [LayerStrategy.HONEYCOMB, LayerStrategy.HYBRID, LayerStrategy.GEODESIC]]
ax.legend(handles=legend_patches, loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig('figures/03_3d_layer_stack.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/03_3d_layer_stack.png')

## 6. Unified Thesis Pipeline Diagram

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(18, 8))
ax.set_xlim(0, 18)
ax.set_ylim(0, 8)
ax.axis('off')

ax.text(9, 7.6, 'Unified Thesis: AI-Driven Autonomous Bioprinting Pipeline',
        ha='center', va='center', fontsize=14, fontweight='bold')

# Phase 1 (Honeycomb)
phase1_y = 5.8
ax.add_patch(plt.Rectangle((0.5, phase1_y - 0.8), 5.5, 1.6,
             facecolor='#FFF3E0', edgecolor='#FF8C00', linewidth=2, zorder=1))
ax.text(3.25, phase1_y + 0.5, 'Phase 1: Honeycomb (Tec de Monterrey)',
        ha='center', fontsize=10, fontweight='bold', color='#E65100')
for x, label in [(1.3, 'Multi-View\nEncoder'), (3.25, 'Polar Decoder\n(3D Layered)'), (5.2, 'Honeycomb\nTSP Toolpath')]:
    ax.text(x, phase1_y - 0.2, label, ha='center', va='center', fontsize=7.5, color='#FF8C00', fontweight='bold')
ax.annotate('', xy=(2.3, phase1_y-0.2), xytext=(1.8, phase1_y-0.2), arrowprops=dict(arrowstyle='->', color='#FF8C00', lw=1.5))
ax.annotate('', xy=(4.3, phase1_y-0.2), xytext=(3.8, phase1_y-0.2), arrowprops=dict(arrowstyle='->', color='#FF8C00', lw=1.5))

# Phase 2 (LiveMesh)
ax.add_patch(plt.Rectangle((6.5, phase1_y - 0.8), 5.5, 1.6,
             facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2, zorder=1))
ax.text(9.25, phase1_y + 0.5, 'Phase 2: LiveMesh (MIT Collaboration)',
        ha='center', fontsize=10, fontweight='bold', color='#0D47A1')
for x, label in [(7.3, 'DeepCurrents\nReconstruction'), (9.25, 'Geodesic\nToolpath'), (11.2, 'Closed-Loop\nDepth Control')]:
    ax.text(x, phase1_y - 0.2, label, ha='center', va='center', fontsize=7.5, color='#1565C0', fontweight='bold')
ax.annotate('', xy=(8.3, phase1_y-0.2), xytext=(7.8, phase1_y-0.2), arrowprops=dict(arrowstyle='->', color='#1565C0', lw=1.5))
ax.annotate('', xy=(10.3, phase1_y-0.2), xytext=(9.8, phase1_y-0.2), arrowprops=dict(arrowstyle='->', color='#1565C0', lw=1.5))

# Shared
ax.add_patch(plt.Rectangle((12.5, phase1_y - 0.8), 5.0, 1.6,
             facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2, zorder=1))
ax.text(15.0, phase1_y + 0.5, 'Shared: Perception + Robot',
        ha='center', fontsize=10, fontweight='bold', color='#1B5E20')
for x, label in [(13.5, 'Depth Sensor\n(RealSense D405)'), (15.0, 'Volumetric\nFusion'), (16.5, '6-DOF Robot\nExecution')]:
    ax.text(x, phase1_y - 0.2, label, ha='center', va='center', fontsize=7.5, color='#2E7D32', fontweight='bold')

# Multi-Strategy (this repo) — center stage
ms_y = 3.0
ax.add_patch(plt.Rectangle((2.0, ms_y - 1.2), 14.0, 2.4,
             facecolor='#F3E5F5', edgecolor='#6A1B9A', linewidth=2.5, zorder=1))
ax.text(9.0, ms_y + 0.9, 'NOVELTY: Multi-Strategy Toolpath Router (This Repository)',
        ha='center', fontsize=11, fontweight='bold', color='#4A148C')

ms_items = [
    (3.5, 'Volumetric\nDecoder Output', '#7B1FA2'),
    (6.5, 'Strategy\nRouter (AI)', '#7B1FA2'),
    (9.0, 'HONEYCOMB\n(deep layers)', '#FF8C00'),
    (11.5, 'GEODESIC\n(surface)', '#0077BE'),
    (14.0, 'HYBRID\n(transition)', '#8B008B'),
]
for x, label, color in ms_items:
    ax.text(x, ms_y - 0.1, label, ha='center', va='center', fontsize=8.5, color=color, fontweight='bold')

ax.annotate('', xy=(5.5, ms_y), xytext=(4.5, ms_y), arrowprops=dict(arrowstyle='->', color='#7B1FA2', lw=2))
for target_x in [9.0, 11.5, 14.0]:
    ax.annotate('', xy=(target_x - 0.8, ms_y), xytext=(7.5, ms_y),
                arrowprops=dict(arrowstyle='->', color='#7B1FA2', lw=1.5, connectionstyle='arc3,rad=0.1'))

# Connecting arrows
ax.annotate('', xy=(5.0, ms_y + 1.3), xytext=(5.0, phase1_y - 0.9),
            arrowprops=dict(arrowstyle='->', color='#FF8C00', lw=1.5, linestyle='dashed'))
ax.annotate('', xy=(9.25, ms_y + 1.3), xytext=(9.25, phase1_y - 0.9),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=1.5, linestyle='dashed'))

ax.text(9.0, 0.8, 'Biological Rationale: tissue regeneration requires structural scaffold (deep) +\n'
        'conformal surface matching (top) — just as bone heals from periosteum inward',
        ha='center', va='center', fontsize=8.5, color='#555', style='italic',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#FAFAFA', edgecolor='#DDD'))

plt.tight_layout()
plt.savefig('figures/04_unified_thesis_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/04_unified_thesis_pipeline.png')

## 7. Strategy Comparison: Metrics & Trade-offs

In [ ]:
# Simulated metrics for each strategy (based on bioprinting literature)
metrics = {
    'Honeycomb': {
        'structural_strength': 0.92,
        'surface_conformity': 0.45,
        'fill_speed': 0.88,
        'material_efficiency': 0.75,
        'cell_viability': 0.70,
    },
    'Geodesic': {
        'structural_strength': 0.55,
        'surface_conformity': 0.95,
        'fill_speed': 0.60,
        'material_efficiency': 0.90,
        'cell_viability': 0.92,
    },
    'Multi-Strategy (Ours)': {
        'structural_strength': 0.89,
        'surface_conformity': 0.91,
        'fill_speed': 0.78,
        'material_efficiency': 0.85,
        'cell_viability': 0.88,
    },
}

categories = ['Structural\nStrength', 'Surface\nConformity', 'Fill\nSpeed',
              'Material\nEfficiency', 'Cell\nViability']
N = len(categories)
angles_radar = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles_radar += angles_radar[:1]

colors_radar = {'Honeycomb': '#FF8C00', 'Geodesic': '#0077BE', 'Multi-Strategy (Ours)': '#8B008B'}

fig = plt.figure(figsize=(14, 6))

# Radar chart
ax_polar = fig.add_subplot(121, projection='polar')
for name, vals in metrics.items():
    values = list(vals.values()) + [list(vals.values())[0]]
    ax_polar.plot(angles_radar, values, 'o-', linewidth=2, label=name,
                  color=colors_radar[name], markersize=6)
    ax_polar.fill(angles_radar, values, alpha=0.1, color=colors_radar[name])

ax_polar.set_xticks(angles_radar[:-1])
ax_polar.set_xticklabels(categories, fontsize=8)
ax_polar.set_ylim(0, 1.0)
ax_polar.set_title('Strategy Comparison\n(Radar Chart)', fontsize=11, fontweight='bold', pad=20)
ax_polar.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)

# Bar comparison
ax = fig.add_subplot(122)
x_pos = np.arange(N)
width = 0.25
for i, (name, vals) in enumerate(metrics.items()):
    values = list(vals.values())
    offset = (i - 1) * width
    ax.bar(x_pos + offset, values, width, label=name,
           color=colors_radar[name], alpha=0.8, edgecolor='k', linewidth=0.5)

ax.set_ylabel('Score (0-1)')
ax.set_title('Strategy Trade-offs\n(Multi-Strategy balances both)', fontsize=11, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([c.replace('\n', ' ') for c in categories], fontsize=8, rotation=15)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('figures/05_strategy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '='*60)
print('ALL MULTI-STRATEGY VISUALIZATIONS GENERATED')
print('='*60)
print('Figures saved:')
print('  01_strategy_classification.png  - Layer classification heatmap')
print('  02_per_layer_toolpath.png       - Per-layer toolpath with actual waypoints')
print('  03_3d_layer_stack.png           - 3D vertical layer stack')
print('  04_unified_thesis_pipeline.png  - All 3 repos in one diagram')
print('  05_strategy_comparison.png      - Radar + bar comparison chart')